# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** one line item

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
print(lines)

assert len(lines) == 4
assert lines['qty'].sum() == 7

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
vendors = pd.DataFrame(vendors_json)
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)
print(joined)

unmatched = joined[joined['_merge'] == 'left_only']
print("Unmatched vendor(s):", unmatched['vendor_id'].unique())
print("Units attached to unmatched vendor:", unmatched['qty'].sum())

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only
Unmatched vendor(s): ['V-07']
Units attached to unmatched vendor: 3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
# Keep the unmatched vendor as its own visible row instead of dropping it.
# Dropping it would silently lose 3 of the 7 units; a labeled row keeps the total at 7.
joined['name'] = joined['name'].fillna('UNMATCHED (' + joined['vendor_id'] + ')')
qty_by_vendor = joined.groupby('name')['qty'].sum()
print(qty_by_vendor)
assert qty_by_vendor.sum() == 7

name
Cav Merch           1
Hoos Burgers        3
UNMATCHED (V-07)    3
Name: qty, dtype: int64


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
# dropna=False keeps the missing-zone rows visible instead of letting groupby discard them
zone_totals = joined.groupby('zone', dropna=False)['qty'].sum()
print(zone_totals)

matched_total = joined.loc[joined['zone'].notna(), 'qty'].sum()
print("Total with a known zone:", matched_total)
print("Overall total:", joined['qty'].sum())

zone
A      4
NaN    3
Name: qty, dtype: int64
Total with a known zone: 4
Overall total: 7


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
# explode turns each list element into its own row; an order with no 'lines' becomes a NaN row (kept, not crashed on)
df = pd.DataFrame(ragged_json).explode('lines', ignore_index=True)

# expand each line dict into columns; non-dict (missing) lines become empty dicts -> NaN columns
detail = pd.json_normalize(df['lines'].apply(lambda x: x if isinstance(x, dict) else {}).tolist())

ragged = pd.concat([df.drop(columns='lines'), detail], axis=1)

# nullable integer: missing stays <NA>, a real 0 would stay 0
ragged['qty'] = ragged['qty'].astype('Int64')
print(ragged)
print(ragged.dtypes)
print("Rows with missing qty:\n", ragged[ragged['qty'].isna()])

   order vendor_id   item   qty
0      4      V-01   Soda     2
1      5      V-10    NaN  <NA>
2      6      V-01  Fries  <NA>
order         int64
vendor_id    object
item         object
qty           Int64
dtype: object
Rows with missing qty:
    order vendor_id   item   qty
1      5      V-10    NaN  <NA>
2      6      V-01  Fries  <NA>


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

A)

One row per line item is the useful shape because the line item is the smallest unit of what was sold. Questions like total quantity per item, per vendor, or per zone become simple groupby and sum operations. With one row per order, the line items would sit in a nested list inside a single cell, so you couldn't aggregate by item or sum quantities without unpacking them first. It also keeps the vendor join clean: every row has one vendor_id, so a left join keeps the row count at 4 and any mismatch (like V-07) shows up as a visible row rather than being hidden.

B)

A quantity of zero is a known fact: the item was on the order and none were sold or ordered. A missing quantity means the value is unknown, so we don't know how many there were. If you conflate them with fillna(0), the unknown becomes a false fact. Totals get understated with no warning, averages are pulled down, and data-quality problems (like the Fries line with no qty) disappear instead of being flagged for follow-up. Keeping missing as NaN/<NA> lets you count how many values are missing, exclude them deliberately, or go back to the source to fix them. The same applies to order 5 with no lines: it is not an order of zero items, it is an order with no line data.